# ATML PA3 – GPU runner (Colab / Kaggle)

Thin driver only: every experiment is a module in the repo. `outputs/` (checkpoints, merged models, caches) and
`results/` are symlinked to persistent storage, and every script resumes / skips finished work when re-run.

Rough budget: Task 1 needs five 1,250-update runs (Full FT on a T4 is the slowest; an L4/A100 is strongly
recommended) plus seven evaluation passes; Task 2 one 4,500-update run; Task 3 is CPU except the 50-question live
experiment.

In [1]:
import torch
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

Tesla T4 15.6 GB


In [2]:
REPO_URL = "https://github.com/ahmedmahni07-commits/27100150_PA3.git"

import os
ON_KAGGLE = os.path.exists("/kaggle/working")
WORK    = "/kaggle/working" if ON_KAGGLE else "/content"
PERSIST = "/kaggle/working/persist" if ON_KAGGLE else "/content/drive/MyDrive/ATML_PA3"
if not ON_KAGGLE:
    from google.colab import drive
    drive.mount("/content/drive")
for d in ["outputs", "results", "data_large"]:
    os.makedirs(f"{PERSIST}/{d}", exist_ok=True)
print("Kaggle" if ON_KAGGLE else "Colab", "->", PERSIST)

Kaggle -> /kaggle/working/persist


In [3]:
%cd $WORK
!rm -rf pa3 && git clone -q $REPO_URL pa3
%cd $WORK/pa3
!rm -rf outputs && ln -s $PERSIST/outputs outputs
!mkdir -p $PERSIST/results && cp -rn results/. $PERSIST/results/ ; rm -rf results && ln -s $PERSIST/results results
# large data folders live on Drive too
!for d in task2_raw task2_images ATML-PA3-Data; do mkdir -p $PERSIST/data_large/$d; ln -sfn $PERSIST/data_large/$d data/$d; done
!pip -q install -r requirements.txt
!python -m tests.test_task1 && python -m tests.test_task3

/kaggle/working
/kaggle/working/pa3
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pathos 0.3.5 requires dill>=0.4.1, but you have dill 0.4.0 which is incompatible.
pathos 0.3.5 requires multiprocess>=0.70.19, but you have multiprocess 0.70.16 which is incompatible.
s3fs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but 

## Task 2 – VLM

In [4]:
!python -m task2_vlm.data download
!python -m task2_vlm.data select
!python -m task2_vlm.data images
!python -m task2_vlm.features --config configs/task2.yaml

downloading http://images.cocodataset.org/annotations/annotations_trainval2014.zip
downloading https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Train_mscoco.zip
downloading https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Val_mscoco.zip
downloading https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Train_mscoco.zip
downloading https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Val_mscoco.zip
downloading https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Complementary_Pairs_Train_mscoco.zip
downloading https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Complementary_Pairs_Val_mscoco.zip
['captions_train2014.json', 'captions_val2014.json', 'instances_train2014.json', 'instances_val2014.json', 'person_keypoints_train2014.json', 'person_keypoints_val2014.json', 'v2_OpenEnded_mscoco_train2014_questions.json', 'v2_OpenEnded_mscoco_val2014_questions.json', 'v2_mscoco_train2014_annotations.json', 'v2_mscoco_train2014_complementary_pairs.json', 'v2_mscoco_val

In [5]:
!python -m task2_vlm.train --config configs/task2.yaml

tokenizer_config.json: 9.73kB [00:00, 32.2MB/s]
vocab.json: 2.78MB [00:00, 87.7MB/s]
merges.txt: 1.67MB [00:00, 124MB/s]
tokenizer.json: 100%|██████████████████████| 11.4M/11.4M [00:00<00:00, 14.7MB/s]
config.json: 100%|█████████████████████████████| 726/726 [00:00<00:00, 5.03MB/s]
I0000 00:00:1793790318.010628     256 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
model.safetensors: 100%|████████████████████| 1.50G/1.50G [00:03<00:00, 466MB/s]
generation_config.json: 100%|██████████████████| 239/239 [00:00<00:00, 1.83MB/s]
dev@0 {'dev_caption_loss': 5.464216709136963, 'grounding_dev_nll_correct': 8.573830604553223, 'grounding_dev_nll_swap': 8.344776153564453, 'grounding_dev_nll_none': 3.513364791870117, 'grounding_dev_swap_penalty': -0.229054793715477, 'grounding_dev_noimg_penalty': -5.060466289520264, 'vqa_dev_nll_correct': 11.230072021484375, 'vqa_dev_nll_swap': 10.925494194030762, 'vqa_dev_nll_none': 4.5412139892578125, 'vqa_dev_swap_penalty':

In [6]:
!python -m task2_vlm.evaluate --config configs/task2.yaml
!python -m task2_vlm.summarize --config configs/task2.yaml

I0000 00:00:1793797777.123285     315 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
    suite  condition  accuracy  pair_accuracy  identical_answer_rate  empty_rate  follows_partner_rate  mean_ref_nll  gen_ms_per_answer   n  acc_counting  acc_presence  acc_number  acc_other  acc_yes/no
grounding    correct     70.31          57.81                  20.31         0.0                 10.16          0.81              45.14 128         46.88         93.75         NaN        NaN         NaN
grounding mismatched     37.50           7.81                  62.50         0.0                 28.91          2.11              47.76 128         20.31         54.69         NaN        NaN         NaN
grounding       none     36.72           0.00                 100.00         0.0                 36.72          1.30              17.52 128         23.44         50.00         NaN        NaN         NaN
grounding       swap     10.16           1.56                  2

In [7]:
# optional 49-token run
# !python -m task2_vlm.train --config configs/task2.yaml --n-vis 49
# !python -m task2_vlm.evaluate --config configs/task2.yaml --n-vis 49

In [8]:
import os, json, pandas as pd
p = "/kaggle/working/pa3/results/task2/vis196/train_log.jsonl"
print("found" if os.path.exists(p) else "NOT FOUND - this session doesn't have the run's files")
if os.path.exists(p):
    t = pd.DataFrame([json.loads(l) for l in open(p)])
    print(t.groupby("stage")["seconds"].sum().div(3600).round(3))
    print("total h:", round(t["seconds"].sum() / 3600, 3))

found
stage
caption     0.897
instruct    0.757
vqa         0.372
Name: seconds, dtype: float64
total h: 2.027


In [9]:
%cd /kaggle/working/pa3
!zip -qr /kaggle/working/task2_results.zip results/task2 data/task2 outputs/task2/vis196/stage*.pt
!ls -lh /kaggle/working/task2_results.zip

from IPython.display import FileLink
FileLink("/kaggle/working/task2_results.zip")

/kaggle/working/pa3
-rw-r--r-- 1 root root 54M Nov  4 13:11 /kaggle/working/task2_results.zip


/kaggle/working/task2_results.zip

## Task 3 – test-time compute (copy ATML-PA3-Data into data/ATML-PA3-Data first)

In [10]:
   %cd /kaggle/working/pa3
   !git pull
   !ls results/task3/live_problems.json
   !python -m task3_ttc.live generate
   !python -m task3_ttc.live summarize
   !zip -qr /kaggle/working/task3_live.zip results/task3/live
   from IPython.display import FileLink
   FileLink("/kaggle/working/task3_live.zip")

/kaggle/working/pa3
Already up to date.
results/task3/live_problems.json
tokenizer_config.json: 7.30kB [00:00, 25.5MB/s]
vocab.json: 2.78MB [00:00, 97.1MB/s]
merges.txt: 1.67MB [00:00, 125MB/s]
tokenizer.json: 7.03MB [00:00, 154MB/s]
config.json: 100%|█████████████████████████████| 661/661 [00:00<00:00, 5.16MB/s]
I0000 00:00:1793797888.917616     365 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
model.safetensors.index.json: 35.6kB [00:00, 80.8MB/s]
Fetching 2 files:   0%|                                   | 0/2 [00:00<?, ?it/s]
model-00002-of-00002.safetensors:   0%|             | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|             | 0.00/3.97G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%| | 20.3k/3.97G [00:01<55:54:02, 19.7kB/s]
model-00002-of-00002.safetensors:   1%|    | 25.1M/2.20G [00:01<01:29, 24.3MB/s]
model-00002-of-00002.safetensors:   4%|▏   | 90.8M/2.20G [00:01<00:24, 85.9MB/s]
model-00002-of-000

/kaggle/working/task3_live.zip

In [11]:
!python -m task3_ttc.live generate
!python -m task3_ttc.live summarize
!python -m task3_ttc.plots

I0000 00:00:1793806328.879556     472 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Loading checkpoint shards: 100%|██████████████████| 2/2 [00:03<00:00,  1.96s/it]
0 test:943 done
1 test:2688 done
2 test:3504 done
3 test:4335 done
4 test:2460 done
5 test:1324 done
6 test:2340 done
7 test:1577 done
8 test:639 done
9 test:2892 done
10 test:4702 done
11 test:1709 done
12 test:4041 done
13 test:2119 done
14 test:2087 done
15 test:4549 done
16 test:2732 done
17 test:1433 done
18 test:3285 done
19 test:29 done
20 test:2948 done
21 test:1997 done
22 test:414 done
23 test:1808 done
24 test:4962 done
25 test:274 done
26 test:916 done
27 test:534 done
28 test:2177 done
29 test:1043 done
30 test:373 done
31 test:1379 done
32 test:196 done
33 test:2134 done
34 test:771 done
35 test:3572 done
36 test:2358 done
37 test:2019 done
38 test:4144 done
39 test:3314 done
40 test:3239 done
41 test:4678 done
42 test:1122 done
43 test:4168 done
44 test:1300 done
45 tes

In [12]:
!ls -la /kaggle/working/
!ls -la /kaggle/working/pa3/results/task3/live/

total 54472
drwxr-xr-x  4 root root     4096 Nov  4 15:32 .
drwxr-xr-x  8 root root     4096 Nov  4 10:54 ..
-rw-r--r--  1 root root    75311 Nov  4 15:32 __notebook__.ipynb
drwxr-xr-x 11 root root     4096 Nov  4 10:54 pa3
drwxr-xr-x  5 root root     4096 Nov  4 10:54 persist
-rw-r--r--  1 root root 55589842 Nov  4 13:11 task2_results.zip
-rw-r--r--  1 root root    92376 Nov  4 15:32 task3_live.zip
total 716
drwxr-xr-x 2 root root   4096 Nov  4 15:32 .
drwxr-xr-x 7 root root   4096 Nov  4 13:11 ..
-rw-r--r-- 1 root root 692220 Nov  4 15:31 calls.jsonl
-rw-r--r-- 1 root root  13848 Nov  4 15:32 per_question.csv
-rw-r--r-- 1 root root    348 Nov  4 15:32 summary.csv
-rw-r--r-- 1 root root    128 Nov  4 15:32 transitions.json
-rw-r--r-- 1 root root   1371 Nov  4 15:32 transitions_per_question.csv
